## Basic tutorial of MSFT Agent Framework

In [1]:
import os
import asyncio
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential

In [2]:
from dotenv import load_dotenv

load_dotenv('.env')
OPENAI_API_ENDPOINT = os.getenv("OPENAI_API_ENDPOINT")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

### Simple Agent chat completion

In [8]:
# Initiate chat client. Note that AzureOpenAIChatClient uses AzureCliCredential by default if endpoint
# and api_key are not provided. The deployment_name is required.
joker_agent = AzureOpenAIChatClient(
    deployment_name="gpt-5-mini",
    endpoint=OPENAI_API_ENDPOINT,
    api_key=OPENAI_API_KEY
).create_agent(
    instructions="You are good at telling jokes.",
    name="Joker"
)

In [ ]:
# Single chat completion run:
async def main():
    result = await joker_agent.run("Tell me a joke about a pirate.")
    print(result.text)

await main()

What did the ocean say to the pirate? Nothing — it just waved.


In [ ]:
# Multiple concurrent chat completions:
prompts = [
    "Joke about pirates",
    "Joke about astronauts",
    "Joke about quantum physicists"
]
results = await asyncio.gather(*(joker_agent.run(p) for p in prompts))
for r in results:
    print(r.text)

Why did the pirate go to school? To improve his arrrrrithmetic.

You might think a pirate's favorite letter is R — but really, it's the C.
Why did the astronaut break up with his girlfriend? He needed space.
A cop stops a car with Heisenberg and Schrödinger inside. He asks Heisenberg, “Do you know how fast you were going?” Heisenberg says, “No, but I know exactly where I am.” The cop opens the trunk, finds a cat, and asks, “Did you know you had a dead cat in here?” Schrödinger replies, “Well, now we do.”


In [ ]:
# Using ChatMessage can allow for more complex messages, including images:
from agent_framework import ChatMessage, TextContent, UriContent, Role
image_url = "https://fastly.picsum.photos/id/26/4209/2769.jpg?hmac=vcInmowFvPCyKGtV7Vfh7zWcA_Z0kStrPDW3ppP0iGI"

message = ChatMessage(
    role=Role.USER,
    contents=[
        TextContent(text="Describe the following image:"),
        UriContent(uri=image_url, media_type="image/jpeg")
    ]
)

img_agent = AzureOpenAIChatClient(deployment_name="gpt-5-mini", endpoint=OPENAI_API_ENDPOINT, api_key=OPENAI_API_KEY
    ).create_agent(
        instructions="You are good at analyzing images.",
        name="Image Describer"
)

result = await agent.run(message)
print(result.text)

A neatly arranged flat‑lay of everyday men’s accessories on a gray textured surface. The items are organized in a grid and include:

- A small black zip wallet (top left) and a black felt/leather cardholder (bottom left).
- Gold‑rim aviator sunglasses (upper center left) and a gray tweed bow tie (upper center).
- A silver ballpoint pen (center) with a small red lacquered case or box beneath it.
- A stack of slim notebooks (top right).
- A black smartphone (middle left), a wristwatch with a light face and patterned strap (middle center), and a pair of black rectangular reading glasses (middle right).
- A pair of black over‑ear headphones with a coiled cable and gold connector (right).

The color palette is muted and slightly vintage, with black leather, metallic accents, gray fabric, and a touch of red. The overall look is organized and stylish — everyday carry items laid out for display.


In [ ]:
# We can also load local files as context:
from agent_framework import DataContent

with open("landmarks.jpg", "rb") as f:
    image_bytes = f.read()

message = ChatMessage(
    role=Role.USER,
    contents=[
        TextContent(text="What do you see in this image?"),
        DataContent(data=image_bytes, media_type="image/jpeg")
    ]
)

result = await img_agent.run(message)
print(result.text)

A collage of famous French landmarks and sights. From left to right, top to bottom:

- Top left: the Eiffel Tower (Champ de Mars), Paris  
- Top middle: Notre‑Dame Cathedral, Paris  
- Top right (top row): the Arc de Triomphe (and the Palace of Versailles interior at the far right)  
- Middle right/top far right: an interior/court of the Palace of Versailles  
- Bottom left: Pont Alexandre III (illuminated at dusk)  
- Bottom middle-left: Mont‑Saint‑Michel (with its reflection)  
- Bottom middle: the glass pyramid at the Louvre Museum  
- Bottom right: the Paris Catacombs (walls of skulls/bones)

Overall it’s a montage of iconic Paris/France tourist attractions.


### Multi-turn conversations with Agent Framework

Regular ChatCompletion objects are stateless and do not maintain any state internally between calls. To have a multi-turn conversation with an agent, you need to create an object to hold the conversation state and pass this object to the agent when running it.

In [ ]:
# Pass threads to the joker agent (note that it's possible to pass multiple threads to the same agent to maintain
# different conversations):
thread = joker_agent.get_new_thread()

async def main():
    result1 = await joker_agent.run("Tell me a joke about a pirate.", thread=thread)
    print(result1.text)

    result2 = await joker_agent.run("Now add some emojis to the joke and tell it in the voice of a pirate's parrot.", thread=thread)
    print(result2.text)

    result3 = await joker_agent.run("Now pretend how would the pirate respond to the previous joke?", thread=thread)
    print(result3.text)

await main()

Why did the pirate go to school? To improve his arrrrr-ticulation!
Squawk! Why did th' pirate go t' school? Squawk! To improve his arrrrr-ticulation! Arrr! 🦜🏴‍☠️📚😂
Arrr! Ye blasted bird, that be a fine jest — it made me timbers shiver! Keep squawkin' like that and I'll name ye First Mate o' Funny. 😂🏴‍☠️🦜


In [ ]:
# Threads have the seralize method to export the thread as a dictionary, which can then be saved as JSON.
# JSON threads can be reloaded and resumed later:
import json

# Serialize the thread state:
serialized_thread = await thread.serialize()
serialized_json = json.dumps(serialized_thread)
with open('file_path', "w") as f:
    f.write(serialized_json)

# Read the JSON and deserialize the thread state to be used again:
with open('file_path.json', "r") as f:
    reloaded_data = json.loads(f.read())
resumed_thread = await joker_agent.deserialize_thread(reloaded_data)

### Structured outputs with Pydantic models
We can use the `response_format` parameter to specify the desired output schema, defined by a Pydantic model schema.

In [4]:
from pydantic import BaseModel

In [5]:
# Define Pydantic data class:
class PersonInfo(BaseModel):
    """ Information about a person. """
    name: str | None = None
    age: int | None = None
    occupation: str | None = None

In [6]:
# Create agent and call it with the response_format parameter to specify the desired output schema:
info_structure_agent = AzureOpenAIChatClient(deployment_name="gpt-5-mini", endpoint=OPENAI_API_ENDPOINT, api_key=OPENAI_API_KEY
    ).create_agent(
        name="HelpfulAssistant",
        instructions="You are a helpful assistant that extracts person information from text."
)

response = await info_structure_agent.run(
    "Please provide information about John Smith, who is a 35-year-old software engineer.",
    response_format=PersonInfo
)
print(response.text)

{"name":"John Smith","age":35,"occupation":"software engineer"}
